# Debug: Data Quality & Join Validation

This notebook validates all pricing tables and checks for **cost calculation readiness**.

## Checks Performed

| Check | Description |
|-------|-------------|
| 1 | `instance_rates` ↔ `vm_costs` - All instances have VM prices |
| 2 | `dbsql_warehouse_config` ↔ `vm_costs` - SQL warehouse VMs have prices |
| 3 | `databricks_regions` ↔ `vm_costs` - All regions have VM prices |
| 4 | `dbu_prices` ↔ `sku_region_mapping` - All DBU regions are mapped |
| 5-7 | Rate tables ↔ `dbu_prices` - SKU types exist in pricing |
| 8 | Cloud coverage summary |
| 9 | Cloud naming consistency (AWS, AZURE, GCP) |
| **10** | `dbsql_rates` ↔ `dbu_prices` - SQL SKU types joinable |
| **11** | Classic compute SKU types in `dbu_prices` |
| **12** | Region coverage - VM costs vs DBU prices |
| **13** | 🎯 **COST CALCULATION READINESS** - All workloads ready? |

## Tables Checked
`databricks_regions`, `instance_rates`, `vm_costs`, `dbsql_rates`, `dbsql_warehouse_config`, `dbu_prices`, `sku_region_mapping`, `dbu_multipliers`, `serverless_product_rates`, `fmapi_databricks_rates`, `fmapi_proprietary_rates`


In [ ]:
# Configuration
CATALOG = "lakemeter_catalog"
SCHEMA = "lakemeter"

# Table names
TABLES = {
    "regions": f"{CATALOG}.{SCHEMA}.databricks_regions",
    "instance_rates": f"{CATALOG}.{SCHEMA}.instance_rates",
    "vm_costs": f"{CATALOG}.{SCHEMA}.vm_costs",
    "dbsql_rates": f"{CATALOG}.{SCHEMA}.dbsql_rates",
    "dbsql_warehouse_config": f"{CATALOG}.{SCHEMA}.dbsql_warehouse_config",
    "dbu_prices": f"{CATALOG}.{SCHEMA}.dbu_prices",
    "sku_region_mapping": f"{CATALOG}.{SCHEMA}.sku_region_mapping",
    "dbu_multipliers": f"{CATALOG}.{SCHEMA}.dbu_multipliers",
    "serverless_product_rates": f"{CATALOG}.{SCHEMA}.serverless_product_rates",
    "fmapi_databricks_rates": f"{CATALOG}.{SCHEMA}.fmapi_databricks_rates",
    "fmapi_proprietary_rates": f"{CATALOG}.{SCHEMA}.fmapi_proprietary_rates",
}

print("✅ Configuration loaded")
print(f"📊 Will check {len(TABLES)} tables")


In [ ]:
# =============================================================================
# TABLE EXISTENCE CHECK
# =============================================================================

print("="*60)
print("📋 TABLE EXISTENCE CHECK")
print("="*60)

existing_tables = []
missing_tables = []

for name, table in TABLES.items():
    try:
        count = spark.sql(f"SELECT COUNT(*) as cnt FROM {table}").collect()[0]['cnt']
        existing_tables.append(name)
        print(f"✅ {name}: {count:,} rows")
    except Exception as e:
        missing_tables.append(name)
        print(f"❌ {name}: TABLE NOT FOUND")

print(f"\n📊 Summary: {len(existing_tables)} tables exist, {len(missing_tables)} missing")
if missing_tables:
    print(f"⚠️ Missing tables: {missing_tables}")


In [ ]:
# =============================================================================
# CHECK 1: instance_rates ↔ vm_costs
# All instance types in instance_rates should have VM costs
# Note: Strip "(Beta)" suffix from instance names for matching
# =============================================================================

print("="*60)
print("🔍 CHECK 1: instance_rates ↔ vm_costs")
print("   Do all instance types have VM costs?")
print("="*60)

if 'instance_rates' in existing_tables and 'vm_costs' in existing_tables:
    for cloud in ['AWS', 'AZURE', 'GCP']:
        # Strip "(Beta)" suffix for matching - AWS beta instances don't have "(Beta)" in pricing API
        query = f"""
        SELECT ir.instance_type, ir.cloud,
               TRIM(REPLACE(ir.instance_type, '(Beta)', '')) as clean_instance_type
        FROM {TABLES['instance_rates']} ir
        LEFT JOIN {TABLES['vm_costs']} vc 
            ON UPPER(TRIM(REPLACE(ir.instance_type, '(Beta)', ''))) = UPPER(vc.instance_type) 
            AND UPPER(ir.cloud) = UPPER(vc.cloud)
        WHERE UPPER(ir.cloud) = '{cloud}'
        AND vc.instance_type IS NULL
        """
        missing = spark.sql(query).toPandas()
        
        # Separate beta and non-beta
        beta_missing = missing[missing['instance_type'].str.contains('Beta', na=False)]
        non_beta_missing = missing[~missing['instance_type'].str.contains('Beta', na=False)]
        
        if len(non_beta_missing) > 0:
            print(f"\n⚠️ {cloud}: {len(non_beta_missing)} instance types WITHOUT VM costs:")
            for _, row in non_beta_missing.head(20).iterrows():
                print(f"   - {row['instance_type']}")
            if len(non_beta_missing) > 20:
                print(f"   ... and {len(non_beta_missing) - 20} more")
        else:
            print(f"✅ {cloud}: All non-beta instance types have VM costs")
        
        if len(beta_missing) > 0:
            print(f"   ℹ️ {len(beta_missing)} Beta instances (may not have public pricing yet)")
else:
    print("⏭️ Skipped - required tables missing")


In [ ]:
# =============================================================================
# CHECK 1b: Deep dive on Azure missing instances
# Show what's in vm_costs vs instance_rates for debugging
# =============================================================================

print("="*60)
print("🔍 CHECK 1b: Azure Instance Type Analysis")
print("="*60)

if 'instance_rates' in existing_tables and 'vm_costs' in existing_tables:
    # Get Azure instances from instance_rates
    ir_azure = spark.sql(f"""
        SELECT DISTINCT instance_type 
        FROM {TABLES['instance_rates']} 
        WHERE UPPER(cloud) = 'AZURE'
    """).toPandas()['instance_type'].tolist()
    
    # Get Azure instances from vm_costs
    vc_azure = spark.sql(f"""
        SELECT DISTINCT instance_type 
        FROM {TABLES['vm_costs']} 
        WHERE UPPER(cloud) = 'AZURE'
    """).toPandas()['instance_type'].tolist()
    
    print(f"📋 instance_rates has {len(ir_azure)} Azure instance types")
    print(f"📋 vm_costs has {len(vc_azure)} Azure instance types")
    
    # Check for pattern differences
    # Many Azure instances in vm_costs use format like "Standard_D16a_v4" 
    # while instance_rates might have "Standard_D16a_v4" or similar
    
    # Show sample of what's in vm_costs
    print(f"\n📋 Sample vm_costs Azure instances (first 20):")
    for inst in sorted(vc_azure)[:20]:
        print(f"   - {inst}")
    
    # Show missing ones and try to find close matches
    missing_azure = [i for i in ir_azure if i not in vc_azure]
    print(f"\n📋 Missing from vm_costs ({len(missing_azure)}):")
    for inst in sorted(missing_azure)[:30]:
        # Try to find close match
        close = [v for v in vc_azure if inst.lower().replace('_', '').replace('-', '') in v.lower().replace('_', '').replace('-', '')]
        match_str = f" → possible match: {close[0]}" if close else ""
        print(f"   - {inst}{match_str}")
else:
    print("⏭️ Skipped - required tables missing")


In [ ]:
# =============================================================================
# CHECK 2: dbsql_warehouse_config ↔ vm_costs
# Driver/Worker instance types should have VM costs
# =============================================================================

print("="*60)
print("🔍 CHECK 2: dbsql_warehouse_config ↔ vm_costs")
print("   Do DBSQL driver/worker instances have VM costs?")
print("="*60)

if 'dbsql_warehouse_config' in existing_tables and 'vm_costs' in existing_tables:
    for cloud in ['AWS', 'AZURE', 'GCP']:
        # Check driver instances
        query_driver = f"""
        SELECT DISTINCT dwc.driver_instance_type as instance_type
        FROM {TABLES['dbsql_warehouse_config']} dwc
        LEFT JOIN {TABLES['vm_costs']} vc 
            ON UPPER(dwc.driver_instance_type) = UPPER(vc.instance_type) 
            AND UPPER(dwc.cloud) = UPPER(vc.cloud)
        WHERE UPPER(dwc.cloud) = '{cloud}'
        AND vc.instance_type IS NULL
        """
        
        # Check worker instances
        query_worker = f"""
        SELECT DISTINCT dwc.worker_instance_type as instance_type
        FROM {TABLES['dbsql_warehouse_config']} dwc
        LEFT JOIN {TABLES['vm_costs']} vc 
            ON UPPER(dwc.worker_instance_type) = UPPER(vc.instance_type) 
            AND UPPER(dwc.cloud) = UPPER(vc.cloud)
        WHERE UPPER(dwc.cloud) = '{cloud}'
        AND vc.instance_type IS NULL
        """
        
        missing_driver = spark.sql(query_driver).toPandas()
        missing_worker = spark.sql(query_worker).toPandas()
        
        if len(missing_driver) > 0 or len(missing_worker) > 0:
            print(f"\n⚠️ {cloud}:")
            if len(missing_driver) > 0:
                print(f"   Driver instances missing VM costs: {missing_driver['instance_type'].tolist()}")
            if len(missing_worker) > 0:
                print(f"   Worker instances missing VM costs: {missing_worker['instance_type'].tolist()}")
        else:
            print(f"✅ {cloud}: All DBSQL instances have VM costs")
else:
    print("⏭️ Skipped - required tables missing")


In [ ]:
# =============================================================================
# CHECK 3: sku_region_mapping ↔ vm_costs (dbu_prices = source of truth)
# All regions with DBU prices should have VM costs for cost calculation
# =============================================================================

print("="*60)
print("🔍 CHECK 3: sku_region_mapping ↔ vm_costs")
print("   Do all DBU price regions have VM costs?")
print("   (sku_region_mapping = source of truth from dbu_prices)")
print("="*60)

if 'sku_region_mapping' in existing_tables and 'vm_costs' in existing_tables:
    for cloud in ['AWS', 'AZURE', 'GCP']:
        query = f"""
        SELECT srm.region_code
        FROM {TABLES['sku_region_mapping']} srm
        LEFT JOIN (
            SELECT DISTINCT UPPER(cloud) as cloud, LOWER(region) as region 
            FROM {TABLES['vm_costs']}
        ) vc 
            ON UPPER(srm.cloud) = vc.cloud 
            AND LOWER(srm.region_code) = vc.region
        WHERE UPPER(srm.cloud) = '{cloud}'
        AND vc.region IS NULL
        """
        missing = spark.sql(query).toPandas()
        
        if len(missing) > 0:
            print(f"\n❌ {cloud}: {len(missing)} regions WITHOUT VM costs:")
            for _, row in missing.head(20).iterrows():
                print(f"   - {row['region_code']}")
        else:
            print(f"✅ {cloud}: All DBU price regions have VM costs")
else:
    print("⏭️ Skipped - required tables missing")


In [ ]:
# =============================================================================
# CHECK 4: dbu_prices ↔ sku_region_mapping
# All regions in dbu_prices should have region mappings
# =============================================================================

print("="*60)
print("🔍 CHECK 4: dbu_prices ↔ sku_region_mapping")
print("   Do all DBU price regions have mappings?")
print("="*60)

if 'dbu_prices' in existing_tables and 'sku_region_mapping' in existing_tables:
    # 1. Regions in dbu_prices but NOT in mapping (by cloud)
    query_missing = f"""
    SELECT DISTINCT dp.region, dp.cloud
    FROM {TABLES['dbu_prices']} dp
    LEFT JOIN {TABLES['sku_region_mapping']} srm
        ON dp.region = srm.sku_region
        AND UPPER(dp.cloud) = UPPER(srm.cloud)
    WHERE srm.sku_region IS NULL
    AND dp.region IS NOT NULL
    AND dp.region != ''
    ORDER BY dp.cloud, dp.region
    """
    missing = spark.sql(query_missing).toPandas()
    
    if len(missing) > 0:
        print(f"\n⚠️ {len(missing)} region(s) in dbu_prices WITHOUT mapping:")
        for cloud in ['AWS', 'AZURE', 'GCP']:
            cloud_missing = missing[missing['cloud'] == cloud]
            if len(cloud_missing) > 0:
                print(f"\n   {cloud} ({len(cloud_missing)}):")
                for _, row in cloud_missing.iterrows():
                    print(f"      - {row['region']}")
    else:
        print("✅ All DBU price regions have mappings")
    
    # 2. Show ALL unique regions in dbu_prices by cloud
    print("\n" + "-"*60)
    print("📊 All unique regions in dbu_prices by cloud:")
    for cloud in ['AWS', 'AZURE', 'GCP']:
        regions_df = spark.sql(f"""
            SELECT DISTINCT region FROM {TABLES['dbu_prices']} 
            WHERE cloud = '{cloud}' AND region IS NOT NULL AND region != ''
            ORDER BY region
        """).toPandas()
        print(f"\n   {cloud} ({len(regions_df)} regions):")
        for r in regions_df['region'].tolist():
            print(f"      - {r}")
    
    # 3. Show ALL unique regions in sku_region_mapping by cloud
    print("\n" + "-"*60)
    print("📊 All unique sku_regions in mapping table by cloud:")
    for cloud in ['AWS', 'AZURE', 'GCP']:
        mapping_df = spark.sql(f"""
            SELECT DISTINCT sku_region FROM {TABLES['sku_region_mapping']} 
            WHERE UPPER(cloud) = '{cloud}'
            ORDER BY sku_region
        """).toPandas()
        print(f"\n   {cloud} ({len(mapping_df)} mappings):")
        for r in mapping_df['sku_region'].tolist():
            print(f"      - {r}")
    
    # 4. Regions in mapping but NOT in dbu_prices (unused mappings)
    print("\n" + "-"*60)
    query_unused = f"""
    SELECT DISTINCT srm.sku_region, srm.cloud
    FROM {TABLES['sku_region_mapping']} srm
    LEFT JOIN {TABLES['dbu_prices']} dp
        ON srm.sku_region = dp.region
        AND UPPER(srm.cloud) = UPPER(dp.cloud)
    WHERE dp.region IS NULL
    ORDER BY srm.cloud, srm.sku_region
    """
    unused = spark.sql(query_unused).toPandas()
    if len(unused) > 0:
        print(f"ℹ️ {len(unused)} mapping(s) NOT used by dbu_prices (extra mappings, OK):")
        for cloud in ['AWS', 'AZURE', 'GCP']:
            cloud_unused = unused[unused['cloud'] == cloud]
            if len(cloud_unused) > 0:
                print(f"\n   {cloud} ({len(cloud_unused)}):")
                for _, row in cloud_unused.iterrows():
                    print(f"      - {row['sku_region']}")
    else:
        print("ℹ️ All mappings are used by dbu_prices")
else:
    print("⏭️ Skipped - required tables missing")


In [ ]:
# =============================================================================
# CHECK 5: serverless_product_rates ↔ dbu_prices
# SKU product types should exist in dbu_prices
# =============================================================================

print("="*60)
print("🔍 CHECK 5: serverless_product_rates ↔ dbu_prices")
print("   Do serverless SKU types exist in pricing table?")
print("="*60)

if 'serverless_product_rates' in existing_tables and 'dbu_prices' in existing_tables:
    # Get unique product types from serverless rates
    spr_types = spark.sql(f"""
        SELECT DISTINCT sku_product_type FROM {TABLES['serverless_product_rates']}
    """).toPandas()['sku_product_type'].tolist()
    
    # Get unique product types from dbu_prices
    dbu_types = spark.sql(f"""
        SELECT DISTINCT product_type FROM {TABLES['dbu_prices']}
    """).toPandas()['product_type'].tolist()
    
    missing = [t for t in spr_types if t not in dbu_types]
    
    if missing:
        print(f"\n⚠️ {len(missing)} SKU types NOT in dbu_prices:")
        for t in missing:
            print(f"   - {t}")
    else:
        print("✅ All serverless SKU types exist in pricing table")
    
    print(f"\n📋 Serverless SKU types: {spr_types}")
else:
    print("⏭️ Skipped - required tables missing")


In [ ]:
# =============================================================================
# CHECK 6: fmapi_databricks_rates ↔ dbu_prices
# SKU product types should exist in dbu_prices
# =============================================================================

print("="*60)
print("🔍 CHECK 6: fmapi_databricks_rates ↔ dbu_prices")
print("   Do FMAPI Databricks SKU types exist in pricing table?")
print("="*60)

if 'fmapi_databricks_rates' in existing_tables and 'dbu_prices' in existing_tables:
    fmapi_types = spark.sql(f"""
        SELECT DISTINCT sku_product_type FROM {TABLES['fmapi_databricks_rates']}
    """).toPandas()['sku_product_type'].tolist()
    
    dbu_types = spark.sql(f"""
        SELECT DISTINCT product_type FROM {TABLES['dbu_prices']}
    """).toPandas()['product_type'].tolist()
    
    missing = [t for t in fmapi_types if t not in dbu_types]
    
    if missing:
        print(f"\n⚠️ {len(missing)} SKU types NOT in dbu_prices:")
        for t in missing:
            print(f"   - {t}")
    else:
        print("✅ All FMAPI Databricks SKU types exist in pricing table")
    
    print(f"\n📋 FMAPI Databricks SKU types: {fmapi_types}")
else:
    print("⏭️ Skipped - required tables missing")


In [ ]:
# =============================================================================
# CHECK 7: fmapi_proprietary_rates ↔ dbu_prices
# SKU product types should exist in dbu_prices
# =============================================================================

print("="*60)
print("🔍 CHECK 7: fmapi_proprietary_rates ↔ dbu_prices")
print("   Do FMAPI Proprietary SKU types exist in pricing table?")
print("="*60)

if 'fmapi_proprietary_rates' in existing_tables and 'dbu_prices' in existing_tables:
    prop_types = spark.sql(f"""
        SELECT DISTINCT sku_product_type FROM {TABLES['fmapi_proprietary_rates']}
    """).toPandas()['sku_product_type'].tolist()
    
    dbu_types = spark.sql(f"""
        SELECT DISTINCT product_type FROM {TABLES['dbu_prices']}
    """).toPandas()['product_type'].tolist()
    
    missing = [t for t in prop_types if t not in dbu_types]
    
    if missing:
        print(f"\n⚠️ {len(missing)} SKU types NOT in dbu_prices:")
        for t in missing:
            print(f"   - {t}")
    else:
        print("✅ All FMAPI Proprietary SKU types exist in pricing table")
    
    print(f"\n📋 FMAPI Proprietary SKU types: {prop_types}")
else:
    print("⏭️ Skipped - required tables missing")


In [ ]:
# =============================================================================
# CHECK 8: dbu_multipliers & dbsql_rates - Cloud coverage
# All clouds should have multipliers and rates
# =============================================================================

print("="*60)
print("🔍 CHECK 8: Cloud Coverage Summary")
print("   Do all clouds have data in key tables?")
print("="*60)

if 'dbu_multipliers' in existing_tables:
    print("\n📋 dbu_multipliers by cloud:")
    df = spark.sql(f"""
        SELECT cloud, COUNT(DISTINCT feature) as features, COUNT(*) as total_rows
        FROM {TABLES['dbu_multipliers']}
        GROUP BY cloud ORDER BY cloud
    """).toPandas()
    display(df)

if 'dbsql_rates' in existing_tables:
    print("\n📋 dbsql_rates by cloud:")
    df = spark.sql(f"""
        SELECT cloud, warehouse_type, COUNT(*) as rows
        FROM {TABLES['dbsql_rates']}
        GROUP BY cloud, warehouse_type ORDER BY cloud, warehouse_type
    """).toPandas()
    display(df.pivot(index='warehouse_type', columns='cloud', values='rows').fillna(0))

if 'dbsql_warehouse_config' in existing_tables:
    print("\n📋 dbsql_warehouse_config by cloud:")
    df = spark.sql(f"""
        SELECT cloud, warehouse_type, COUNT(*) as rows
        FROM {TABLES['dbsql_warehouse_config']}
        GROUP BY cloud, warehouse_type ORDER BY cloud, warehouse_type
    """).toPandas()
    display(df.pivot(index='warehouse_type', columns='cloud', values='rows').fillna(0))


In [ ]:
# =============================================================================
# CHECK 9: Cloud naming consistency
# All tables should use consistent cloud names (AWS, AZURE, GCP)
# =============================================================================

print("="*60)
print("🔍 CHECK 9: Cloud Naming Consistency")
print("   Are cloud names consistent across all tables?")
print("="*60)

expected_clouds = {'AWS', 'AZURE', 'GCP'}
inconsistencies = []

tables_with_cloud = [
    ('regions', 'cloud'),
    ('instance_rates', 'cloud'),
    ('vm_costs', 'cloud'),
    ('dbsql_rates', 'cloud'),
    ('dbsql_warehouse_config', 'cloud'),
    ('dbu_prices', 'cloud'),
    ('sku_region_mapping', 'cloud'),
    ('dbu_multipliers', 'cloud'),
    ('serverless_product_rates', 'cloud'),
    ('fmapi_databricks_rates', 'cloud'),
    ('fmapi_proprietary_rates', 'cloud'),
]

for table_name, col in tables_with_cloud:
    if table_name in existing_tables:
        try:
            clouds = spark.sql(f"SELECT DISTINCT UPPER({col}) as cloud FROM {TABLES[table_name]}").toPandas()['cloud'].tolist()
            clouds = [c for c in clouds if c]  # Remove None
            unexpected = set(clouds) - expected_clouds
            if unexpected:
                inconsistencies.append((table_name, unexpected))
                print(f"⚠️ {table_name}: unexpected values {unexpected}")
            else:
                print(f"✅ {table_name}: {sorted(clouds)}")
        except Exception as e:
            print(f"❌ {table_name}: Error - {e}")

if not inconsistencies:
    print("\n✅ All tables use consistent cloud naming (AWS, AZURE, GCP)")
else:
    print(f"\n⚠️ {len(inconsistencies)} table(s) have naming inconsistencies")


In [ ]:
# =============================================================================
# CHECK 10: dbsql_rates ↔ dbu_prices (NEW - sku_product_type join)
# SQL warehouse rates should have matching DBU prices
# =============================================================================

print("="*60)
print("🔍 CHECK 10: dbsql_rates ↔ dbu_prices")
print("   Do DBSQL SKU types exist in pricing table?")
print("="*60)

if 'dbsql_rates' in existing_tables and 'dbu_prices' in existing_tables:
    # Get unique sku_product_types from dbsql_rates
    dbsql_types = spark.sql(f"""
        SELECT DISTINCT sku_product_type FROM {TABLES['dbsql_rates']}
    """).toPandas()['sku_product_type'].tolist()
    
    # Get unique product_types from dbu_prices
    dbu_types = spark.sql(f"""
        SELECT DISTINCT product_type FROM {TABLES['dbu_prices']}
    """).toPandas()['product_type'].tolist()
    
    missing = [t for t in dbsql_types if t not in dbu_types]
    
    if missing:
        print(f"\n⚠️ {len(missing)} DBSQL SKU types NOT in dbu_prices:")
        for t in missing:
            print(f"   - {t}")
    else:
        print("✅ All DBSQL SKU types exist in pricing table")
    
    print(f"\n📋 DBSQL SKU types: {dbsql_types}")
    
    # Also check cloud × sku_product_type × region coverage
    print("\n📊 Cross-check: dbsql_rates cloud/sku combinations in dbu_prices:")
    query = f"""
    SELECT DISTINCT r.cloud, r.sku_product_type,
           CASE WHEN p.product_type IS NOT NULL THEN '✅' ELSE '❌' END as has_price
    FROM {TABLES['dbsql_rates']} r
    LEFT JOIN (
        SELECT DISTINCT cloud, product_type FROM {TABLES['dbu_prices']}
    ) p ON r.sku_product_type = p.product_type AND UPPER(r.cloud) = UPPER(p.cloud)
    ORDER BY r.cloud, r.sku_product_type
    """
    display(spark.sql(query).toPandas())
else:
    print("⏭️ Skipped - required tables missing")


In [ ]:
# =============================================================================
# CHECK 11: instance_rates ↔ dbu_prices (Classic Compute DBU pricing)
# Classic compute (Jobs, All-Purpose) needs DBU prices
# =============================================================================

print("="*60)
print("🔍 CHECK 11: Classic Compute DBU Pricing")
print("   Do classic compute SKU types exist in dbu_prices?")
print("="*60)

if 'dbu_prices' in existing_tables:
    # Classic compute product types we need for cost calculation
    CLASSIC_COMPUTE_TYPES = [
        "ALL_PURPOSE_COMPUTE",
        "ALL_PURPOSE_COMPUTE_(PHOTON)",
        "ALL_PURPOSE_COMPUTE_(DLT)",
        "JOBS_COMPUTE",
        "JOBS_COMPUTE_(PHOTON)",
        "JOBS_LIGHT_COMPUTE",
        "DLT_CORE_COMPUTE",
        "DLT_CORE_COMPUTE_(PHOTON)",
        "DLT_PRO_COMPUTE",
        "DLT_PRO_COMPUTE_(PHOTON)",
        "DLT_ADVANCED_COMPUTE",
        "DLT_ADVANCED_COMPUTE_(PHOTON)",
    ]
    
    dbu_types = spark.sql(f"""
        SELECT DISTINCT product_type FROM {TABLES['dbu_prices']}
    """).toPandas()['product_type'].tolist()
    
    print("📋 Classic Compute SKU coverage:")
    for sku in CLASSIC_COMPUTE_TYPES:
        status = "✅" if sku in dbu_types else "❌"
        print(f"   {status} {sku}")
    
    missing = [t for t in CLASSIC_COMPUTE_TYPES if t not in dbu_types]
    if missing:
        print(f"\n⚠️ {len(missing)} classic compute types missing from dbu_prices")
    else:
        print(f"\n✅ All {len(CLASSIC_COMPUTE_TYPES)} classic compute types found in dbu_prices")
else:
    print("⏭️ Skipped - dbu_prices table missing")


In [ ]:
# =============================================================================
# CHECK 12: Region coverage - vm_costs ↔ dbu_prices (via sku_region_mapping)
# All regions with VM costs should have DBU prices
# =============================================================================

print("="*60)
print("🔍 CHECK 12: Region Coverage - VM Costs vs DBU Prices")
print("   Do all VM cost regions have DBU prices?")
print("="*60)

if 'vm_costs' in existing_tables and 'dbu_prices' in existing_tables and 'sku_region_mapping' in existing_tables:
    for cloud in ['AWS', 'AZURE', 'GCP']:
        # Get regions with VM costs
        vm_regions = spark.sql(f"""
            SELECT DISTINCT region FROM {TABLES['vm_costs']} WHERE UPPER(cloud) = '{cloud}'
        """).toPandas()['region'].tolist()
        
        # Get regions with DBU prices (mapped to region_code)
        dbu_regions = spark.sql(f"""
            SELECT DISTINCT srm.region_code 
            FROM {TABLES['dbu_prices']} dp
            JOIN {TABLES['sku_region_mapping']} srm 
                ON dp.region = srm.sku_region AND UPPER(dp.cloud) = UPPER(srm.cloud)
            WHERE UPPER(dp.cloud) = '{cloud}'
        """).toPandas()['region_code'].tolist()
        
        # Find regions in vm_costs but not in dbu_prices
        vm_only = set(vm_regions) - set(dbu_regions)
        dbu_only = set(dbu_regions) - set(vm_regions)
        
        print(f"\n📋 {cloud}:")
        print(f"   VM costs regions: {len(vm_regions)}")
        print(f"   DBU prices regions: {len(dbu_regions)}")
        
        if vm_only:
            print(f"   ⚠️ VM costs but NO DBU prices: {sorted(vm_only)}")
        if dbu_only:
            print(f"   ℹ️ DBU prices but NO VM costs: {sorted(dbu_only)}")
        if not vm_only and not dbu_only:
            print(f"   ✅ Perfect region match!")
else:
    print("⏭️ Skipped - required tables missing")


In [ ]:
# =============================================================================
# CHECK 13: COST CALCULATION READINESS SUMMARY
# Can we calculate costs for all workload types?
# =============================================================================

print("="*60)
print("🎯 CHECK 13: COST CALCULATION READINESS")
print("   Can we calculate costs for all workload types?")
print("="*60)

print("\n📋 WORKLOAD TYPE → REQUIRED TABLES → STATUS")
print("-" * 60)

workload_checks = []

# 1. Jobs / All-Purpose Compute (Classic)
check1 = all([
    'instance_rates' in existing_tables,
    'vm_costs' in existing_tables,
    'dbu_prices' in existing_tables,
    'dbu_multipliers' in existing_tables,
])
status1 = "✅ READY" if check1 else "❌ MISSING"
print(f"\n1️⃣ Jobs / All-Purpose (Classic)")
print(f"   Tables: instance_rates + vm_costs + dbu_prices + dbu_multipliers")
print(f"   Status: {status1}")

# 2. DBSQL Classic/Pro
check2 = all([
    'dbsql_rates' in existing_tables,
    'dbsql_warehouse_config' in existing_tables,
    'vm_costs' in existing_tables,
    'dbu_prices' in existing_tables,
])
status2 = "✅ READY" if check2 else "❌ MISSING"
print(f"\n2️⃣ SQL Warehouse (Classic/Pro)")
print(f"   Tables: dbsql_rates + dbsql_warehouse_config + vm_costs + dbu_prices")
print(f"   Status: {status2}")

# 3. DBSQL Serverless
check3 = all([
    'dbsql_rates' in existing_tables,
    'dbu_prices' in existing_tables,
])
status3 = "✅ READY" if check3 else "❌ MISSING"
print(f"\n3️⃣ SQL Warehouse (Serverless)")
print(f"   Tables: dbsql_rates + dbu_prices")
print(f"   Status: {status3}")

# 4. Serverless Products (Vector Search, Model Serving, etc.)
check4 = all([
    'serverless_product_rates' in existing_tables,
    'dbu_prices' in existing_tables,
])
status4 = "✅ READY" if check4 else "❌ MISSING"
print(f"\n4️⃣ Serverless Products (Vector Search, Model Serving, etc.)")
print(f"   Tables: serverless_product_rates + dbu_prices")
print(f"   Status: {status4}")

# 5. FMAPI Databricks Models
check5 = all([
    'fmapi_databricks_rates' in existing_tables,
    'dbu_prices' in existing_tables,
])
status5 = "✅ READY" if check5 else "❌ MISSING"
print(f"\n5️⃣ FMAPI Databricks Models (Llama, etc.)")
print(f"   Tables: fmapi_databricks_rates + dbu_prices")
print(f"   Status: {status5}")

# 6. FMAPI Proprietary Models
check6 = all([
    'fmapi_proprietary_rates' in existing_tables,
    'dbu_prices' in existing_tables,
])
status6 = "✅ READY" if check6 else "❌ MISSING"
print(f"\n6️⃣ FMAPI Proprietary Models (OpenAI, Anthropic, Google)")
print(f"   Tables: fmapi_proprietary_rates + dbu_prices")
print(f"   Status: {status6}")

# Summary
all_ready = all([check1, check2, check3, check4, check5, check6])
print("\n" + "="*60)
if all_ready:
    print("🎉 ALL WORKLOAD TYPES READY FOR COST CALCULATION!")
else:
    print("⚠️ SOME WORKLOAD TYPES NOT READY - check missing tables above")
print("="*60)


In [ ]:
# =============================================================================
# QUICK FIX: Fetch missing r5.2xlarge from AWS Pricing API
# Includes: on_demand, spot, reserved (1y/3y with payment options)
# =============================================================================

print("="*60)
print("🔧 QUICK FIX: Fetch r5.2xlarge from AWS API")
print("="*60)

%pip install boto3 -q

import boto3
import json
import os
import csv
from datetime import datetime, timedelta
import pandas as pd

# Load AWS credentials from secret scope
aws_access_key = dbutils.secrets.get(scope="lakemeter-credentials", key="aws-access-key-id")
aws_secret_key = dbutils.secrets.get(scope="lakemeter-credentials", key="aws-secret-access-key")
os.environ['AWS_ACCESS_KEY_ID'] = aws_access_key
os.environ['AWS_SECRET_ACCESS_KEY'] = aws_secret_key

INSTANCE_TYPE = "r5.2xlarge"

# Delete if exists
spark.sql(f"""
    DELETE FROM {TABLES['vm_costs']} 
    WHERE UPPER(cloud) = 'AWS' 
    AND LOWER(instance_type) = '{INSTANCE_TYPE.lower()}'
""")
print(f"✅ Deleted existing {INSTANCE_TYPE} rows (if any)")

# Get AWS regions from databricks_regions
aws_regions_df = spark.sql(f"""
    SELECT DISTINCT region_code FROM {TABLES['regions']} WHERE UPPER(cloud) = 'AWS'
""").toPandas()
AWS_REGIONS = aws_regions_df['region_code'].tolist()

# Region code to AWS location name mapping
REGION_MAP = {
    'us-east-1': 'US East (N. Virginia)', 'us-east-2': 'US East (Ohio)',
    'us-west-1': 'US West (N. California)', 'us-west-2': 'US West (Oregon)',
    'ca-central-1': 'Canada (Central)', 'eu-west-1': 'EU (Ireland)',
    'eu-west-2': 'EU (London)', 'eu-west-3': 'EU (Paris)',
    'eu-central-1': 'EU (Frankfurt)', 'eu-north-1': 'EU (Stockholm)',
    'ap-northeast-1': 'Asia Pacific (Tokyo)', 'ap-northeast-2': 'Asia Pacific (Seoul)',
    'ap-northeast-3': 'Asia Pacific (Osaka)', 'ap-southeast-1': 'Asia Pacific (Singapore)',
    'ap-southeast-2': 'Asia Pacific (Sydney)', 'ap-south-1': 'Asia Pacific (Mumbai)',
    'sa-east-1': 'South America (Sao Paulo)',
}

all_prices = []

# 1. Fetch On-Demand prices
print(f"\n🔄 Fetching on-demand prices for {INSTANCE_TYPE}...")
pricing_client = boto3.client('pricing', region_name='us-east-1')

for region in AWS_REGIONS:
    location = REGION_MAP.get(region)
    if not location:
        continue
    try:
        response = pricing_client.get_products(
            ServiceCode='AmazonEC2',
            Filters=[
                {'Type': 'TERM_MATCH', 'Field': 'instanceType', 'Value': INSTANCE_TYPE},
                {'Type': 'TERM_MATCH', 'Field': 'location', 'Value': location},
                {'Type': 'TERM_MATCH', 'Field': 'operatingSystem', 'Value': 'Linux'},
                {'Type': 'TERM_MATCH', 'Field': 'tenancy', 'Value': 'Shared'},
                {'Type': 'TERM_MATCH', 'Field': 'preInstalledSw', 'Value': 'NA'},
                {'Type': 'TERM_MATCH', 'Field': 'capacitystatus', 'Value': 'Used'},
            ],
            MaxResults=10
        )
        for item in response.get('PriceList', []):
            data = json.loads(item)
            terms = data.get('terms', {})
            
            # On-Demand
            for term_key, term_val in terms.get('OnDemand', {}).items():
                for pd_key, pd_val in term_val.get('priceDimensions', {}).items():
                    price = float(pd_val['pricePerUnit'].get('USD', 0))
                    if price > 0:
                        all_prices.append({
                            "cloud": "AWS", "region": region, "instance_type": INSTANCE_TYPE,
                            "pricing_tier": "on_demand", "cost_per_hour": round(price, 6),
                            "currency": "USD", "source": "AWS Pricing API",
                            "fetched_at": datetime.utcnow().isoformat()
                        })
            
            # Reserved
            for term_key, term_val in terms.get('Reserved', {}).items():
                term_attrs = term_val.get('termAttributes', {})
                lease = term_attrs.get('LeaseContractLength', '')
                purchase_opt = term_attrs.get('PurchaseOption', '')
                
                tier_name = None
                if lease == '1yr':
                    if purchase_opt == 'No Upfront': tier_name = 'reserved_1y_no_upfront'
                    elif purchase_opt == 'Partial Upfront': tier_name = 'reserved_1y_partial_upfront'
                    elif purchase_opt == 'All Upfront': tier_name = 'reserved_1y_all_upfront'
                elif lease == '3yr':
                    if purchase_opt == 'No Upfront': tier_name = 'reserved_3y_no_upfront'
                    elif purchase_opt == 'Partial Upfront': tier_name = 'reserved_3y_partial_upfront'
                    elif purchase_opt == 'All Upfront': tier_name = 'reserved_3y_all_upfront'
                
                if tier_name:
                    for pd_key, pd_val in term_val.get('priceDimensions', {}).items():
                        if 'Hrs' in pd_val.get('unit', ''):
                            price = float(pd_val['pricePerUnit'].get('USD', 0))
                            all_prices.append({
                                "cloud": "AWS", "region": region, "instance_type": INSTANCE_TYPE,
                                "pricing_tier": tier_name, "cost_per_hour": round(price, 6),
                                "currency": "USD", "source": "AWS Pricing API",
                                "fetched_at": datetime.utcnow().isoformat()
                            })
    except Exception as e:
        print(f"   ⚠️ Error fetching {region}: {e}")

print(f"   ✅ Fetched {len(all_prices)} on-demand/reserved prices")

# 2. Fetch Spot prices
print(f"\n🔄 Fetching spot prices for {INSTANCE_TYPE}...")
spot_count = 0
for region in AWS_REGIONS:
    try:
        ec2 = boto3.client('ec2', region_name=region)
        response = ec2.describe_spot_price_history(
            InstanceTypes=[INSTANCE_TYPE],
            ProductDescriptions=['Linux/UNIX'],
            StartTime=datetime.utcnow() - timedelta(days=1),
            MaxResults=10
        )
        prices = [float(p['SpotPrice']) for p in response.get('SpotPriceHistory', [])]
        if prices:
            avg_price = sum(prices) / len(prices)
            all_prices.append({
                "cloud": "AWS", "region": region, "instance_type": INSTANCE_TYPE,
                "pricing_tier": "spot", "cost_per_hour": round(avg_price, 6),
                "currency": "USD", "source": "AWS EC2 Spot API (24h avg)",
                "fetched_at": datetime.utcnow().isoformat()
            })
            spot_count += 1
    except Exception as e:
        pass  # Spot not available in all regions

print(f"   ✅ Fetched {spot_count} spot prices")

# Insert all prices
if all_prices:
    df_fix = pd.DataFrame(all_prices)
    spark_df = spark.createDataFrame(df_fix)
    spark_df.write.mode("append").saveAsTable(TABLES['vm_costs'])
    
    print(f"\n✅ Added {len(all_prices)} rows for {INSTANCE_TYPE}")
    print(f"📊 By pricing tier:")
    print(df_fix.groupby('pricing_tier').size())
else:
    print("❌ No prices fetched!")


In [ ]:
# =============================================================================
# SUMMARY
# =============================================================================

print("="*60)
print("📊 DATA QUALITY SUMMARY")
print("="*60)

print(f"\n✅ Tables checked: {len(existing_tables)}")
if missing_tables:
    print(f"❌ Missing tables: {missing_tables}")

print("\n" + "="*60)
print("📋 ROW COUNTS BY TABLE")
print("="*60)

for name in sorted(existing_tables):
    try:
        count = spark.sql(f"SELECT COUNT(*) as cnt FROM {TABLES[name]}").collect()[0]['cnt']
        print(f"   {name}: {count:,} rows")
    except:
        pass

print("\n🎉 Data quality check complete!")
print("\n💡 Next steps if issues found:")
print("   1. Missing VM costs → Re-run VM cost fetch notebooks")
print("   2. Missing region mappings → Update sku_region_mapping in notebook 08")
print("   3. Cloud naming issues → Standardize to AWS, AZURE, GCP")
print("   4. Missing tables → Run the corresponding load notebook")
